# Vision Transformer (ViT) Implementation for Book Pose Classification

## 專案目標
本筆記本 (Notebook) 實作 **Vision Transformer (ViT)** 模型，用於書本姿態分類任務。
目標是捨棄傳統 CNN (如 ResNet)，利用 **Self-Attention** 機制捕捉全域特徵。

## 任務資訊
- **輸入資料**: COCO 格式標註的圖片 (需裁切 Bounding Box)。
- **類別**: `book`, `reverse`, `backward`, `flat`。
- **模型**: Pre-trained ViT-B/16 (ImageNet weights)。
- **流程**: Data Loading -> Transforms (224x224) -> Transfer Learning -> Fine-tuning -> Evaluation。


In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Check CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# --- 1. Environment Setup ---

# Define Paths
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..')) # Parent directory as root
DATA_DIR = os.path.join(ROOT_DIR, 'dataset', 'for_vit', '2.v4-boy3new.coco')
OUTPUT_DIR = os.path.join(ROOT_DIR, 'vit_project')

# Create Output Directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory created at: {OUTPUT_DIR}')

# Set Random Seed for Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# --- 2. Data Pipeline (COCO Parsing & Custom Dataset) ---

class CocoCropDataset(Dataset):
    """
    Custom Dataset to parse COCO JSON and crop images based on Bounding Boxes.
    If no bbox is present, uses the full image.
    """
    def __init__(self, root_dir, subset, transform=None):
        """
        Args:
            root_dir (str): Path to the dataset root (containing train/valid/test folders).
            subset (str): 'train', 'valid', or 'test'.
            transform (callable, optional): Transform to be applied on a sample.
        """
        self.subset_dir = os.path.join(root_dir, subset)
        self.json_path = os.path.join(self.subset_dir, '_annotations.coco.json')
        self.transform = transform
        
        # Load JSON
        with open(self.json_path, 'r') as f:
            self.coco_data = json.load(f)
        
        # Map Image ID to File Name
        self.images = {img['id']: img['file_name'] for img in self.coco_data['images']}
        
        # Map Category ID to Name
        # Note: JSON has multiple IDs for 'book' (0 and 2). We merge them by name.
        self.id_to_name = {cat['id']: cat['name'] for cat in self.coco_data['categories']}
        
        # Define Target Classes (0-3)
        self.target_classes = ['book', 'backward', 'flat', 'reverse']
        self.class_to_idx = {name: i for i, name in enumerate(self.target_classes)}
        
        # Prepare Annotations List
        self.samples = []
        valid_labels_count = Counter()
        
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            cat_id = ann['category_id']
            
            if img_id not in self.images:
                continue
            
            # Get Category Name and Map to 0-3 index
            cat_name = self.id_to_name.get(cat_id)
            if cat_name not in self.class_to_idx:
                continue # Skip unknown categories if any
            
            label = self.class_to_idx[cat_name]
            bbox = ann.get('bbox', None) # [x, y, w, h]
            
            self.samples.append({
                'image_path': os.path.join(self.subset_dir, self.images[img_id]),
                'label': label,
                'bbox': bbox
            })
            valid_labels_count[cat_name] += 1
            
        print(f'[{subset}] Loaded {len(self.samples)} samples. Distribution: {dict(valid_labels_count)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image_path = sample['image_path']
        label = sample['label']
        bbox = sample['bbox']
        
        # Load Image
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f'Error loading {image_path}: {e}')
            # Return a black image in case of error (robustness)
            image = Image.new('RGB', (224, 224))
        
        # Crop if bbox exists
        if bbox:
            x, y, w, h = bbox
            # Basic boundary checks
            img_w, img_h = image.size
            x = max(0, x)
            y = max(0, y)
            # Ensure crop is valid
            if w > 0 and h > 0:
                image = image.crop((x, y, x + w, y + h))
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [ ]:
# --- Transforms ---
# ViT-B/16 requires 224x224 input.
# We use ImageNet mean/std for normalization as we are using pre-trained weights.

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create Datasets
train_dataset = CocoCropDataset(DATA_DIR, 'train', transform=train_transform)
valid_dataset = CocoCropDataset(DATA_DIR, 'valid', transform=eval_transform)
test_dataset = CocoCropDataset(DATA_DIR, 'test', transform=eval_transform)

# Create DataLoaders
BATCH_SIZE = 32 # Adjust based on GPU VRAM (ViT is heavy)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Class Mapping: {train_dataset.class_to_idx}')

## 3. Model Architecture: Vision Transformer (ViT)

### ViT 核心概念

1.  **Patch Embedding (切片嵌入)**:
    -   不同於 CNN 滑動視窗 (Sliding Window)，ViT 將 2D 圖片切成固定大小的 Patches (例如 16x16)。
    -   每個 Patch 被展平 (Flatten) 並通過線性層映射為向量 (Embedding)。
    -   這些 Patch Embeddings 就像 NLP 中的單詞 (Tokens) 一樣輸入到 Transformer。

2.  **Self-Attention (自注意力機制)**:
    -   Transformer Encoder 利用 Multi-Head Self-Attention 計算每個 Patch 與整張圖其他 Patch 的關聯性。
    -   這使得模型在第一層就能擁有 **全域感受野 (Global Receptive Field)**，而不像 CNN 需要透過深層堆疊才能看到全圖特徵。

3.  **Class Token**:
    -   ViT 在輸入序列前加入一個可學習的 `[CLS]` token。
    -   最後分類時，只取這個 `[CLS]` token 的輸出進行預測，因為它聚合了整張圖片的資訊。


In [ ]:
# --- Model Implementation ---

def get_vit_model(num_classes):
    # Load pre-trained ViT-B/16
    print('Loading ViT-B/16 weights...')
    weights = models.ViT_B_16_Weights.DEFAULT
    model = models.vit_b_16(weights=weights)
    
    # 1. Freeze Feature Extractor (Backbone)
    for param in model.parameters():
        param.requires_grad = False
    
    # 2. Modify the Heads (Classifier)
    # ViT's classifier is stored in 'heads'
    # Input features for ViT-B/16 is 768
    model.heads = nn.Sequential(
        nn.Linear(768, 512),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(512, num_classes)
    )
    
    return model

model = get_vit_model(len(train_dataset.class_to_idx))
model = model.to(device)

# Check trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable Parameters: {trainable_params}')

In [ ]:
# --- 4. Training Loop ---

# Configuration
LEARNING_RATE = 1e-4 # Low LR for fine-tuning
EPOCHS = 15
PATIENCE = 5 # Early Stopping patience

# Optimizer & Loss
# AdamW is generally preferred for Transformers
optimizer = optim.AdamW(model.heads.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

# Training Functions
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    loop = tqdm(loader, leave=False)
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        loop.set_description(f'Loss: {loss.item():.4f}')
        
    return running_loss / len(loader), 100 * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    return running_loss / len(loader), 100 * correct / total


In [ ]:
# --- Main Training Process ---

best_acc = 0.0
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}
save_path = os.path.join(OUTPUT_DIR, 'best_vit.pth')

print('Starting Training...')
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, valid_loader, criterion)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['valid_loss'].append(valid_loss)
    history['valid_acc'].append(valid_acc)
    
    print(f'Epoch [{epoch+1}/{EPOCHS}] ' 
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
          f'Valid Loss: {valid_loss:.4f} Acc: {valid_acc:.2f}%')
    
    # Early Stopping & Checkpoint
    if valid_acc > best_acc:
        best_acc = valid_acc
        patience_counter = 0
        torch.save(model.state_dict(), save_path)
        print(f'>>> New Best Model Saved! (Acc: {valid_acc:.2f}%)')
    else:
        patience_counter += 1
        print(f'Early Stopping Counter: {patience_counter}/{PATIENCE}')
        
    if patience_counter >= PATIENCE:
        print('Early Stopping triggered.')
        break

In [ ]:
# --- 5. Evaluation & Visualization ---

# Load Best Model
model.load_state_dict(torch.load(save_path))
print('Loaded best model for evaluation.')

# Plot Learning Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['valid_loss'], label='Valid Loss')
plt.legend()
plt.title('Loss Curve')

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['valid_acc'], label='Valid Acc')
plt.legend()
plt.title('Accuracy Curve')
plt.savefig(os.path.join(OUTPUT_DIR, 'vit_learning_curves.png'))
plt.show()

# Confusion Matrix on Test Set
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=train_dataset.target_classes, 
            yticklabels=train_dataset.target_classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('ViT Confusion Matrix')
plt.savefig(os.path.join(OUTPUT_DIR, 'vit_confusion_matrix.png'))
plt.show()

print(classification_report(all_labels, all_preds, target_names=train_dataset.target_classes))

## 6. Comparison: CNN vs ViT (Observations)

*(Please fill this section after running the experiment)*

**1. Training Efficiency (訓練效率)**:
- **CNN (ResNet)**: 通常收斂較快，對小資料集較友善。
- **ViT**: 由於缺乏 Inductive Bias (如平移不變性)，通常需要更多資料或更長時間的 Fine-tuning 才能達到穩定效果。觀察 Loss 下降的曲線是否比 ResNet 震盪？

**2. Accuracy (準確度)**:
- 在這個特定資料集上，ViT 的表現如何？是否超越了 ResNet？
- 如果圖片背景複雜，ViT 的 Self-Attention 機制可能有助於忽略背景噪聲，專注於書本本體。

**3. Inference Speed (推論速度)**:
- ViT-B/16 的參數量約為 86M，通常比 ResNet18/50 重。推論速度是否變慢？
